# M2-B1 — Audit de qualité et effet du nettoyage

Les **outils** sont fournis (rapport de qualité, comparaison des modèles). Les **constats**, les **décisions**
et leur **interprétation** sont à toi : réponds dans les cellules ✏️.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")
engine = create_engine(os.environ["DATABASE_URL"])

df = pd.read_sql(text("""
    SELECT h.code AS hotel, r.*
    FROM reservations AS r JOIN hotels AS h ON h.hotel_id = r.hotel_id
    ORDER BY r.arrival_date
"""), engine, parse_dates=["arrival_date", "reservation_status_date"])
df.shape

(78703, 35)

## Tâche 2 — L'audit

In [2]:
# Bornes « raisonnables » par colonne numérique : hypothèses de départ, à discuter
BORNES = {
    "lead_time": (0, 730),
    "adults": (1, 10),
    "children": (0, 10),
    "babies": (0, 5),
    "stays_in_weekend_nights": (0, 20),
    "stays_in_week_nights": (0, 50),
    "adr": (0, 1000),
}


def rapport_qualite(d: pd.DataFrame) -> pd.DataFrame:
    """Manquants et valeurs hors bornes, colonne par colonne."""
    lignes = []
    for col in d.columns:
        info = {"colonne": col, "type": str(d[col].dtype),
                "manquants": int(d[col].isna().sum()), "valeurs_distinctes": d[col].nunique()}
        if col in BORNES:
            bas, haut = BORNES[col]
            info["hors_bornes"] = int(((d[col] < bas) | (d[col] > haut)).sum())
        lignes.append(info)
    return pd.DataFrame(lignes).set_index("colonne")


rapport = rapport_qualite(df)
rapport[(rapport["manquants"] > 0) | (rapport.get("hors_bornes", 0) > 0)]

,type,manquants,valeurs_distinctes,hors_bornes
colonne,,,,
lead_time,int64,0,439,1.0
adults,int64,0,14,285.0
children,float64,4,5,0.0
babies,int64,0,5,2.0
country,object,405,159,NaN
agent,float64,10676,284,NaN
company,float64,74216,298,NaN
adr,float64,0,6985,1.0


In [3]:
# Doublons : réservations identiques sur toutes les colonnes, sauf l'identifiant
doublons = df.drop(columns="reservation_id").duplicated(keep="first")
print(doublons.sum(), "lignes en double")

22999 lignes en double


### ✏️ Ce que le rapport montre

Huit colonnes signalées. `company` manque sur 74 216 lignes et `agent` sur 10 676, mais ni l'une ni l'autre n'entre dans le modèle. Restent `country` sur 405 lignes et `children` sur 4, quatre réservations de l'été 2015 toutes annulées.

Les hors bornes sont rares, sauf `adults` avec ses 285 lignes. En regardant les valeurs, il y a deux choses différentes dans la même colonne : 273 réservations à zéro adulte, et 12 réservations de 20 à 55 personnes. Je les traiterai séparément. Pour le reste, un 9 et un 10 sur `babies`, une nuit à 5 400 €, une réservation faite 737 jours à l'avance. Une ligne chacune.

Les doublons sont le gros morceau, 22 999 lignes soit 29,2 % de la base. Elles annulent à 63,3 % contre 25 % pour les lignes uniques. Ce ne sont pas des erreurs pour autant. 10 572 viennent du segment Groups et la base n'a pas de colonne « nombre de chambres », donc un car de 180 personnes s'écrit en 180 lignes identiques. Les supprimer effacerait les trois quarts des réservations de groupe. J'ai vérifié qu'aucun groupe identique n'est à cheval sur la bascule du 31 août 2016, il n'y a pas de fuite de ce côté.

### ✏️ Ce que le rapport ne voit pas

Le rapport ne connaît que des bornes colonne par colonne. Cherche les **incohérences métier** : des combinaisons
de valeurs impossibles ou suspectes pour un hôtel. Au moins deux, chiffrées. Écris le code dans les cellules
ci-dessous.



In [4]:
# Deux colonnes calculees qui servent a tous les croisements
df["nuits"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["occupants"] = df["adults"] + df["children"].fillna(0) + df["babies"]

# Chaque combinaison suspecte, avec son volume et son taux d'annulation
CAS = {
    "aucune nuit": df["nuits"] == 0,
    "aucun adulte": df["adults"] == 0,
    "aucun occupant": df["occupants"] == 0,
    "prix nul": df["adr"] == 0,
    "prix nul sur un sejour a honorer": (df["adr"] == 0) & (df["nuits"] > 0),
    "plus de 10 occupants": df["occupants"] > 10,
    "acompte non remboursable": df["deposit_type"] == "Non Refund",
    "client fidele sans historique": (df["is_repeated_guest"] == 1)
        & (df["previous_cancellations"] + df["previous_bookings_not_canceled"] == 0),
}

pd.DataFrame([
    {
        "cas": nom,
        "lignes": int(m.sum()),
        "part de la base %": round(m.mean() * 100, 2),
        "taux annulation %": round(df.loc[m, "is_canceled"].mean() * 100, 1),
        "part des annulations %": round(df.loc[m, "is_canceled"].sum() / df["is_canceled"].sum() * 100, 1),
    }
    for nom, m in CAS.items()
]).set_index("cas")

,lignes,part de la base %,taux annulation %,part des annulations %
cas,,,,
aucune nuit,571,0.73,4.2,0.1
aucun adulte,273,0.35,24.5,0.2
aucun occupant,113,0.14,12.4,0.0
prix nul,1504,1.91,10.7,0.6
prix nul sur un sejour a honorer,933,1.19,14.7,0.5
plus de 10 occupants,14,0.02,92.9,0.0
acompte non remboursable,10461,13.29,99.2,36.4
client fidele sans historique,412,0.52,7.5,0.1


In [5]:
# L'acompte non remboursable, le cas que Karim a signale
display(df.groupby("deposit_type")["is_canceled"].agg(lignes="size", taux_annulation="mean").round(3))

# Par segment, pour voir si un seul canal explique le taux
display(df[df["deposit_type"] == "Non Refund"]
        .groupby("market_segment")["is_canceled"]
        .agg(lignes="size", taux_annulation="mean").round(3))

# Le poids de ces lignes dans le taux global
print(f"taux de la base   : {df['is_canceled'].mean():.1%}")
print(f"hors Non Refund   : {df.loc[df['deposit_type'] != 'Non Refund', 'is_canceled'].mean():.1%}")

,lignes,taux_annulation
deposit_type,,
No Deposit,68107,0.266
Non Refund,10461,0.992
Refundable,135,0.074


,lignes,taux_annulation
market_segment,,
Corporate,146,0.849
Direct,13,0.769
Groups,6395,0.991
Offline TA/TO,3856,0.999
Online TA,51,0.961


taux de la base   : 36.2%
hors Non Refund   : 26.5%


In [6]:
# Prix nul : les gratuites du segment Complementary sont attendues, les autres moins
nul = df["adr"] == 0
print(f"{nul.sum()} reservations a 0 EUR, dont {(nul & (df['market_segment'] == 'Complementary')).sum()} en Complementary")

df.groupby("market_segment")["adr"].agg(lignes="size", prix_median="median").round(1).join(
    df[nul].groupby("market_segment").size().rename("dont prix nul")
)

1504 reservations a 0 EUR, dont 479 en Complementary


,lignes,prix_median,dont prix nul
market_segment,,,
Aviation,127,95.0,2.0
Complementary,529,0.0,479.0
Corporate,3733,65.0,66.0
Direct,7977,99.7,165.0
Groups,13957,65.0,210.0
Offline TA/TO,18552,85.0,295.0
Online TA,33826,104.1,287.0
Undefined,2,15.0,NaN


In [7]:
# A quoi ressemblent les lignes concernees
COLONNES = ["hotel", "arrival_date", "adults", "children", "babies", "nuits", "adr",
            "lead_time", "market_segment", "deposit_type", "reservation_status", "is_canceled"]

print("sans nuit")
display(df.loc[df["nuits"] == 0, COLONNES].head(5))

print("sans adulte")
display(df.loc[df["adults"] == 0, COLONNES].head(5))

print("valeurs extremes")
display(pd.concat([df.nlargest(1, "adr"), df.nlargest(3, "occupants"),
                   df.nlargest(2, "babies"), df.nlargest(1, "lead_time")])[COLONNES])

sans nuit


,hotel,arrival_date,adults,children,babies,nuits,adr,lead_time,market_segment,deposit_type,reservation_status,is_canceled
79,Resort Hotel,2015-07-01,2,0.0,0,0,0.0,342,Direct,No Deposit,Check-Out,0
80,Resort Hotel,2015-07-01,2,0.0,0,0,0.0,737,Direct,No Deposit,Check-Out,0
449,Resort Hotel,2015-07-06,2,0.0,0,0,0.0,111,Online TA,No Deposit,Check-Out,0
450,Resort Hotel,2015-07-06,1,0.0,0,0,0.0,0,Direct,No Deposit,Check-Out,0
509,Resort Hotel,2015-07-07,2,0.0,0,0,0.0,8,Direct,No Deposit,Check-Out,0


sans adulte


,hotel,arrival_date,adults,children,babies,nuits,adr,lead_time,market_segment,deposit_type,reservation_status,is_canceled
1959,City Hotel,2015-07-23,0,0.0,0,7,0.0,132,Groups,No Deposit,Check-Out,0
2251,City Hotel,2015-07-25,0,0.0,0,4,0.0,110,Offline TA/TO,No Deposit,Check-Out,0
3748,City Hotel,2015-08-09,0,0.0,0,2,0.0,0,Online TA,No Deposit,Check-Out,0
3869,City Hotel,2015-08-10,0,3.0,0,2,9.0,1,Direct,No Deposit,Check-Out,0
3905,City Hotel,2015-08-10,0,0.0,0,29,0.0,0,Online TA,No Deposit,Check-Out,0


valeurs extremes


,hotel,arrival_date,adults,children,babies,nuits,adr,lead_time,market_segment,deposit_type,reservation_status,is_canceled
31828,City Hotel,2016-03-25,2,0.0,0,1,5400.00,35,Offline TA/TO,Non Refund,Canceled,1
12428,Resort Hotel,2015-10-04,55,0.0,0,2,0.00,338,Direct,No Deposit,Canceled,1
7716,Resort Hotel,2015-09-07,50,0.0,0,3,0.00,336,Direct,No Deposit,Canceled,1
6989,Resort Hotel,2015-09-03,40,0.0,0,3,0.00,304,Direct,No Deposit,Canceled,1
22709,City Hotel,2016-01-12,2,0.0,10,2,84.45,37,Online TA,No Deposit,Check-Out,0
13759,City Hotel,2015-10-11,1,0.0,9,3,95.00,11,Corporate,No Deposit,Check-Out,0
80,Resort Hotel,2015-07-01,2,0.0,0,0,0.00,737,Direct,No Deposit,Check-Out,0


In [8]:
# Un doublon a cheval sur la bascule train/test ferait voir le test au modele pendant l'entrainement
bascule = df["arrival_date"].max() - pd.DateOffset(months=4)
jeu = (df["arrival_date"] > bascule).map({True: "test", False: "train"})

cles = [c for c in df.columns if c not in ("reservation_id", "hotel_id", "nuits", "occupants")]
groupes = df.assign(jeu=jeu).groupby(cles, dropna=False)["jeu"]
a_cheval = (groupes.transform("size") > 1) & (groupes.transform("nunique") > 1)

print(f"bascule le {bascule.date()}")
print(f"{a_cheval.sum()} lignes dans un groupe identique present des deux cotes")

bascule le 2016-08-31
0 lignes dans un groupe identique present des deux cotes


## Tâche 3 — Les décisions

| Anomalie | Lignes | Décision | Justification | Question au client ? |
|---|---|---|---|---|
| `deposit_type = Non Refund` annulé à 99,2 % | 10 461 | garder et signaler | 36,4 % des annulations de la base. La décision dépend du moment où la valeur est saisie, pas de la donnée elle-même | oui, bloquante |
| Doublons | 22 999 | garder | pas de colonne « nombre de chambres », un groupe s'écrit en autant de lignes, et aucun n'est à cheval sur la bascule train/test | oui |
| `adr = 0` | 1 504 | garder et signaler | 479 viennent du segment Complementary, dont le prix médian est 0 €. Les 1 025 autres ne se distinguent pas d'une saisie oubliée | oui |
| Zéro nuit | 571 | garder et signaler | annulées à 4,2 % et en statut Check-Out, donc des séjours réellement honorés | oui |
| `adults = 0` | 273 | supprimer | une chambre ne se loue pas sans adulte, et `guests` comme `adr_per_guest` se calculent dessus | oui |
| `is_repeated_guest = 1` sans historique | 412 | garder et signaler | hors périmètre : ni cette colonne ni `previous_bookings_not_canceled` ne sont chargées par `train.py` | non |
| `country` manquant | 405 | garder et signaler | le pipeline impute déjà les catégories manquantes à `Unknown`, une règle ferait doublon | non |
| `adults > 10` | 12 | supprimer | de 20 à 55 personnes sur une seule ligne, annulées à 93 %, à 0 € : des pré-réservations de groupe | non |
| `children` manquant | 4 | imputer à 0 | 4 lignes, et la réservation sans enfant est de loin le cas le plus fréquent | non |
| `babies` à 9 et 10 | 2 | garder et signaler | la colonne n'entre pas dans le modèle, l'ajouter à la requête pour deux lignes ne se justifie pas | non |
| `market_segment = Undefined` | 2 | garder | le `OneHotEncoder` à `min_frequency=50` les regroupe déjà avec les modalités rares | non |
| `adr = 5 400 €` | 1 | supprimer | le deuxième prix le plus haut de la base est 508 €, et le `StandardScaler` cale son échelle sur cette valeur | non |
| `lead_time = 737` | 1 | garder | 7 jours au-delà de ma borne, et cette borne est une hypothèse de départ, pas une règle de l'hôtel | non |

### Les questions au client

**1. Le `deposit_type` est-il saisi à la réservation, ou renseigné après coup quand l'annulation est constatée ?**
10 461 réservations, annulées à 99,2 %, soit 36,4 % de toutes les annulations de la base. Si la valeur est connue à la réservation, c'est la variable la plus prédictive dont je dispose. Si elle est écrite après, c'est une fuite de données et la colonne doit sauter, comme les quatre que j'ai retirées en M1-B1. En attendant : je garde la colonne et les lignes, et je le signale dans la fiche.

**2. Une réservation à 0 € est-elle une gratuité commerciale ou un prix non saisi ?**
1 504 lignes, dont 479 dans un segment où 0 € est le tarif normal. En attendant : je garde tout. Supprimer 1 504 lignes dont un tiers sont légitimes coûte plus cher que l'erreur qu'elles introduisent.

**3. La base peut-elle fournir un identifiant de dossier ?**
Sans lui, rien ne distingue un car de 180 personnes d'une double saisie. En attendant : je garde les 22 999 doublons.

**4. L'hôtel vend-il des chambres à la journée ?**
571 réservations à zéro nuit, presque toutes honorées. En attendant : je garde.

**5. Les réservations sans adulte sont-elles une saisie incomplète, ou un type de ligne que l'hôtel connaît ?**
273 lignes, dont 113 sans aucun occupant et une à 29 nuits pour 0 €. En attendant : je les supprime.

### Ce que le nettoyage va retirer

Trois règles suppriment, une impute. Au total 286 lignes, 0,36 % de la base, dont 89 dans le jeu de test. Le taux d'annulation passe de 36,185 % à 36,215 %.

C'est volontairement peu. Les quatre grosses anomalies ne se tranchent pas sans le client. Il faut donc s'attendre à un effet très faible sur le modèle en tâche 6.

## Tâche 6 — Comparer v2.0.0 et v2.1.0

À faire **après** avoir branché le nettoyage et produit la v2.1.0 avec `python train.py`.
La fonction de comparaison est fournie : c'est le **choix du jeu de test** qui t'appartient.

In [10]:
import joblib
from sklearn.metrics import f1_score, precision_score, recall_score

import train as T
from features import INPUT_FEATURES


def comparer(modeles: dict, X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    lignes = []
    for nom, modele in modeles.items():
        pred = modele.predict(X)
        lignes.append({"modèle": nom, "f1": f1_score(y, pred), "rappel": recall_score(y, pred),
                       "précision": precision_score(y, pred), "n": len(y)})
    return pd.DataFrame(lignes).set_index("modèle").round(3)


modeles = {
    "v2.0.0 (données brutes)": joblib.load(ROOT / "model" / "cancellation_model_v2.0.0.joblib"),
    "v2.1.0 (données nettoyées)": joblib.load(ROOT / "model" / "cancellation_model_v2.1.0.joblib"),
}

brut = T.load_data(engine)
_, test_brut = T.split_train_test(brut)
comparer(modeles, test_brut[INPUT_FEATURES], test_brut[T.TARGET])

,f1,rappel,précision,n
modèle,,,,
v2.0.0 (données brutes),0.734,0.754,0.715,19911
v2.1.0 (données nettoyées),0.734,0.754,0.715,19911


In [11]:
from nettoyage import nettoyer

# Le nettoyage s'applique avant le decoupage, comme dans train.py : le test perd 89 lignes
nettoye, _ = nettoyer(brut)
train_nettoye, test_nettoye = T.split_train_test(nettoye)
display(comparer(modeles, test_nettoye[INPUT_FEATURES], test_nettoye[T.TARGET]))

# A 3 decimales les deux modeles se confondent, il faut le 4e chiffre pour voir l'ecart
for nom, jeu in [("brut", test_brut), ("nettoye", test_nettoye)]:
    s0, s1 = (f1_score(jeu[T.TARGET], m.predict(jeu[INPUT_FEATURES])) for m in modeles.values())
    print(f"test {nom:8s} n={len(jeu)} | v2.0.0 {s0:.4f} | v2.1.0 {s1:.4f} | ecart {s1 - s0:+.4f}")

# Sur combien de lignes les deux modeles ne disent pas la meme chose
p0, p1 = (m.predict(test_brut[INPUT_FEATURES]) for m in modeles.values())
print(f"{(p0 != p1).sum()} desaccords sur les {len(test_brut)} lignes du test brut")

,f1,rappel,précision,n
modèle,,,,
v2.0.0 (données brutes),0.735,0.755,0.716,19822
v2.1.0 (données nettoyées),0.735,0.756,0.716,19822


test brut     n=19911 | v2.0.0 0.7341 | v2.1.0 0.7344 | ecart +0.0003
test nettoye  n=19822 | v2.0.0 0.7350 | v2.1.0 0.7353 | ecart +0.0003
55 desaccords sur les 19911 lignes du test brut


In [12]:
from features import ajouter_variables

# Les 89 lignes que le test nettoye ne contient plus : en production l'API peut les recevoir
exclues = test_brut.loc[test_brut.index.difference(test_nettoye.index)]
print(f"{len(exclues)} lignes exclues, annulees a {exclues[T.TARGET].mean():.1%} contre {test_brut[T.TARGET].mean():.1%} sur tout le test")
for nom, modele in modeles.items():
    proba = modele.predict_proba(exclues[INPUT_FEATURES])[:, 1]
    print(f"  {nom} : {int(modele.predict(exclues[INPUT_FEATURES]).sum())} predites annulees, proba moyenne {proba.mean():.2f}")

# guests vaut 0 sur ces lignes, une valeur absente de l'entrainement du v2.1.0
guests = ajouter_variables(train_nettoye[INPUT_FEATURES])["guests"]
print(f"guests appris : min {guests.min():.0f}, moyenne {guests.mean():.2f} | guests=0 est a {(0 - guests.mean()) / guests.std():.1f} ecarts-types")

89 lignes exclues, annulees a 25.8% contre 38.0% sur tout le test
  v2.0.0 (données brutes) : 20 predites annulees, proba moyenne 0.16
  v2.1.0 (données nettoyées) : 18 predites annulees, proba moyenne 0.15
guests appris : min 1, moyenne 1.94 | guests=0 est a -3.1 ecarts-types


### ✏️ Interprétation

**1.** Stable. À jeu de test identique, le F1 gagne +0,0003, sur le brut comme sur le nettoyé, et les deux modèles ne se contredisent que sur 55 des 19 911 lignes du test. Attendu : le nettoyage retire 197 lignes d'un entraînement qui en compte 58 792, et les quatre grosses anomalies sont restées en place, elles attendent les réponses du client.

Le piège est ailleurs. Le même modèle v2.0.0 passe de 0,7341 à 0,7350 en changeant de jeu de test. Le choix du jeu pèse trois fois plus que le changement de modèle. Comparer les deux fiches telles quelles, 0,734 contre 0,735, attribue au nettoyage un gain qui vient d'un test allégé de 89 lignes.

Une baisse ne serait pas forcément une mauvaise nouvelle. Un score mesuré sur un test qui contient des lignes fausses ne veut rien dire de précis. Si le nettoyage retirait des lignes faciles à prédire, le score baisserait en devenant plus sincère.

**2.** 89 lignes, toutes par la règle `adults = 0`. Les 12 groupes et le prix à 5 400 € tombent dans la période d'entraînement. Elles sont annulées à 25,8 %, contre 38,0 % sur le test entier.

En production, le modèle ne les refuse pas. `ajouter_variables` protège la division, donc `guests` vaut 0 et `adr_per_guest` reprend l'`adr`. Il répond, probabilité moyenne 0,15, 18 annulations prédites sur 89. Mais le minimum appris pour `guests` est 1, et 0 se situe à 3,1 écarts-types sous la moyenne : c'est une extrapolation, pas une prédiction appuyée sur des cas comparables.

**3.** Oui, les mêmes règles, mais pas la même action. À l'entraînement, je supprime la ligne. Sur une requête unitaire, l'API n'a rien d'autre à traiter : elle doit refuser l'entrée avec un message clair, `adults` au minimum à 1, plutôt que renvoyer un score que personne ne saura interpréter. Les règles de `nettoyage.py` deviennent des règles de validation d'entrée.

**4.** Le brut ressemble le plus à la production d'aujourd'hui. Les 273 réservations sans adulte sont dans la base de l'hôtel, donc le formulaire les accepte et l'API en recevra.

Je retiens le nettoyé pour décider, à condition que la validation d'entrée de la question 3 soit en place : c'est elle qui rend le flux réel conforme au jeu nettoyé. Ici les deux jeux donnent le même chiffre à 0,001 près, le choix ne change aucune décision, mais il comptera quand le client aura tranché sur les acomptes non remboursables. La règle qui en sortira portera sur 10 461 lignes, pas sur 286.